# FairNet on CelebA

This notebook uses the maintained `fairnet` package. The paper predicts CelebA **Male** (index 20) with **Blond Hair** (index 9) as the binary sensitive attribute.

Prepare the data once with `python scripts/prepare_celeba.py --out data/celeba`, then set `data_root` below. For the scripted, multi-seed reproduction of every paper table see [REPRODUCTION.md](REPRODUCTION.md) and `experiments/`; this notebook is the single-run, readable version of the same pipeline.

In [ ]:
import torch
from transformers import ViTConfig

from fairnet import (
    AttributeMode,
    FairNetConfig,
    FairNetTrainer,
    FairNetViT,
    create_celeba_loaders,
    evaluate_model,
    print_metrics,
    seed_everything,
)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
data_root = "data/celeba"
train_loader, val_loader, test_loader = create_celeba_loaders(
    data_root,
    batch_size=128,
    image_size=64,
    target_attr=20,
    sensitive_attr=9,
    # Supplementary Table 5 reports 162,688 labelled training samples at 100%
    # coverage, which is 162770 // 128 * 128.
    drop_last=True,
)

The following backbone matches Appendix C.2. Reduce the hidden size, layer count, or epochs for a quick CPU smoke test.

In [ ]:
config = FairNetConfig(
    attribute_mode=AttributeMode.FULL,
    sensitive_attributes=[9],
    target_attribute=20,
    detector_layer=4,
    lora_rank=8,
    lora_alpha=16.0,
    lora_layers=[4, 5, 6, 7],
    contrastive_margin=0.5,
    # Stage 1 trains the ViT from scratch, so it needs a real schedule. These
    # are the values in experiments/configs/celeba_base.yaml, selected on
    # validation accuracy.
    stage1_epochs=30,
    stage1_lr=5e-5,
    stage4_epochs=10,
    stage4_lr=1e-4,
    weight_decay=0.05,
    warmup_steps=500,
    lr_schedule="warmup_cosine",
    batch_size=128,
    device=str(device),
)
# Appendix C.2: 8 layers, 8 heads, intermediate size 768, 64x64 inputs, 16x16
# patches. That is 29.57M parameters, matching Supplementary Table F.
vit_config = ViTConfig(
    image_size=64,
    patch_size=16,
    num_channels=3,
    hidden_size=768,
    num_hidden_layers=8,
    num_attention_heads=8,
    intermediate_size=768,
)
model = FairNetViT(vit_config, config)
trainer = FairNetTrainer(model, config, device)

First train and audit the ERM base model. Full mode uses ground-truth sensitive labels as the conditional switch, so no detector MLP is trained.

In [ ]:
trainer.stage1_train_base(train_loader, val_loader)
baseline_metrics = evaluate_model(model, test_loader, config, device, use_lora=False)
print_metrics(baseline_metrics, "ERM test results")

In [ ]:
trainer.stage2_train_detector(train_loader)
trainer.stage3_build_prototypes(train_loader)
trainer.stage4_train_lora(train_loader, val_loader)
fairnet_metrics = evaluate_model(model, test_loader, config, device)
print_metrics(fairnet_metrics, "FairNet-Full test results")

In [ ]:
comparison = {
    metric: (baseline_metrics[metric], fairnet_metrics[metric])
    for metric in ("accuracy", "worst_group_accuracy", "EOD")
}
comparison